# SQL 数据库特性

> **适用场景**: 事务处理、并发控制、数据库维护
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录
1. 事务 ACID & 隔离级别
2. MVCC 多版本并发控制
3. 死锁检测 & 预防
4. Vacuum / Analyze (PostgreSQL)
5. 练习题

---
## 1. 事务 ACID & 隔离级别

### ACID 四大特性

| 特性 | 含义 | 示例 |
|------|------|------|
| **A**tomicity (原子性) | 事务中所有操作要么全成功，要么全回滚 | 转账：扣款+入账必须同时成功 |
| **C**onsistency (一致性) | 事务前后数据满足所有约束条件 | 余额不能为负 |
| **I**solation (隔离性) | 并发事务互不干扰 | 两人同时转账不会互相覆盖 |
| **D**urability (持久性) | 提交后数据永久保存，即使系统崩溃 | WAL 日志保证崩溃恢复 |

### 四种隔离级别 & 解决的问题

| 隔离级别 | 脏读 | 不可重复读 | 幻读 | 性能 |
|----------|------|-----------|------|------|
| READ UNCOMMITTED | ✅可能 | ✅可能 | ✅可能 | 最高 |
| READ COMMITTED | ❌防止 | ✅可能 | ✅可能 | 高 |
| REPEATABLE READ | ❌防止 | ❌防止 | ✅可能* | 中 |
| SERIALIZABLE | ❌防止 | ❌防止 | ❌防止 | 最低 |

*PostgreSQL 的 REPEATABLE READ 也防止幻读（通过 MVCC）

### 三种并发问题
- **脏读 (Dirty Read)**: 读到另一事务**未提交**的数据
- **不可重复读 (Non-repeatable Read)**: 同一事务内两次读取同一行，结果不同（被其他事务 UPDATE）
- **幻读 (Phantom Read)**: 同一事务内两次查询，第二次多/少了行（被其他事务 INSERT/DELETE）

```sql
-- PostgreSQL 设置隔离级别
BEGIN;
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
-- ... 操作 ...
COMMIT;

-- 或全局设置
SET default_transaction_isolation = 'read committed';
```

---
## 2. MVCC 多版本并发控制

### 核心思想
**读不阻塞写，写不阻塞读**。每个事务看到数据的一个**快照版本**，而不是锁住数据等待。

### PostgreSQL MVCC 实现

每行数据有隐藏字段：
- `xmin`: 创建该行版本的事务 ID
- `xmax`: 删除/更新该行的事务 ID（0 表示未删除）

```
初始状态 (xmin=100 提交的事务插入):
┌──────────┬──────┬──────┬─────────┐
│ xmin=100 │ xmax=0 │ id=1 │ val='A' │
└──────────┴──────┴──────┴─────────┘

事务 xid=200 执行 UPDATE val='B':
┌──────────┬────────┬──────┬─────────┐  ← 旧版本（对 xid<200 可见）
│ xmin=100 │ xmax=200 │ id=1 │ val='A' │
└──────────┴────────┴──────┴─────────┘
┌──────────┬──────┬──────┬─────────┐  ← 新版本（对 xid>=200 可见）
│ xmin=200 │ xmax=0 │ id=1 │ val='B' │
└──────────┴──────┴──────┴─────────┘
```

### 可见性规则
事务 T 能看到某行版本，当且仅当：
1. `xmin` 对应事务已提交，且 `xmin < T.snapshot`
2. `xmax` 为 0（未删除），或 `xmax` 对应事务未提交，或 `xmax >= T.snapshot`

### 副作用：表膨胀
旧版本行不会立即删除 → 需要 **VACUUM** 清理死元组

---
## 3. 死锁检测 & 预防

### 什么是死锁？
两个或多个事务互相等待对方持有的锁，形成循环依赖。

```
事务 A: LOCK 行1 → 等待 行2
事务 B: LOCK 行2 → 等待 行1
         ↑________________↑  死锁！
```

### PostgreSQL 死锁检测
- 后台进程定期检查等待图（Wait-for Graph）
- 发现循环后，自动回滚代价最小的事务并报错：`ERROR: deadlock detected`

### 预防策略

**1. 固定加锁顺序**（最有效）
```sql
-- ❌ 危险：不同顺序
-- 事务A: UPDATE accounts WHERE id=1; UPDATE accounts WHERE id=2;
-- 事务B: UPDATE accounts WHERE id=2; UPDATE accounts WHERE id=1;

-- ✅ 安全：统一按 id 升序
-- 事务A: UPDATE accounts WHERE id IN (1,2) ORDER BY id;
-- 事务B: UPDATE accounts WHERE id IN (1,2) ORDER BY id;
```

**2. 使用 `SELECT FOR UPDATE` 提前锁定**
```sql
BEGIN;
SELECT * FROM accounts WHERE id = 1 FOR UPDATE;  -- 提前锁
SELECT * FROM accounts WHERE id = 2 FOR UPDATE;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;
```

**3. 缩短事务时长**，减少锁持有时间

**4. `lock_timeout` 设置**
```sql
SET lock_timeout = '5s';  -- 超过5秒等锁就报错，避免长期阻塞
```

---
## 4. Vacuum / Analyze (PostgreSQL)

### 为什么需要 VACUUM？
MVCC 会保留旧版本行（死元组），导致：
- 表文件不断增大（表膨胀）
- 查询需要跳过大量死元组
- Transaction ID 耗尽风险（XID Wraparound）

### VACUUM vs VACUUM FULL

| 操作 | 作用 | 锁级别 | 适用场景 |
|------|------|--------|----------|
| `VACUUM` | 标记死元组空间可复用，但不归还 OS | 不锁表 | 日常维护 |
| `VACUUM FULL` | 重写整张表，归还磁盘空间给 OS | **排他锁**（锁表） | 表严重膨胀时 |
| `VACUUM ANALYZE` | VACUUM + 更新统计信息 | 不锁表 | 常用组合 |

```sql
-- 手动执行
VACUUM ANALYZE orders;          -- 清理并更新统计信息
VACUUM FULL orders;             -- 完整回收空间（需要维护窗口）

-- 查看表膨胀情况
SELECT relname, n_dead_tup, n_live_tup,
       round(n_dead_tup::numeric / NULLIF(n_live_tup,0) * 100, 2) AS dead_pct
FROM pg_stat_user_tables
ORDER BY n_dead_tup DESC;
```

### ANALYZE
- 收集列的统计信息（分布、distinct 值等）
- 查询优化器依赖统计信息选择执行计划
- 大批量 INSERT/UPDATE 后应手动运行

```sql
ANALYZE orders;  -- 只更新统计信息，不清理死元组
```

### Autovacuum
- PostgreSQL 自动运行 vacuum/analyze
- 触发条件：`autovacuum_vacuum_threshold + autovacuum_vacuum_scale_factor * n_live_tup`
- 高写入表可适当调低 `scale_factor` 让其更频繁触发

---
## 5. 练习题

### Q1 [高频] 解释 ACID 中的隔离性，以及四种隔离级别各解决了什么问题？

<details><summary>参考答案</summary>

隔离性指并发事务之间互不干扰。SQL 标准定义了三类并发问题：
- **脏读**：读到未提交的数据 → READ COMMITTED 解决
- **不可重复读**：同一事务两次读同一行结果不同 → REPEATABLE READ 解决
- **幻读**：同一事务两次查询行数不同 → SERIALIZABLE 解决（PostgreSQL REPEATABLE READ 也解决）

实际生产中 PostgreSQL 默认 READ COMMITTED，高并发 OLTP 可考虑 REPEATABLE READ。
</details>

---

### Q2 [高频] MVCC 如何做到读写不互相阻塞？

<details><summary>参考答案</summary>

MVCC 为每行数据保存多个版本（通过 xmin/xmax 标记）。读操作根据事务快照选择可见的版本，**无需加锁**。写操作创建新版本行，不覆盖旧行。因此读和写操作访问的是不同版本，互不阻塞。

代价：旧版本需要 VACUUM 定期清理，否则表会膨胀。
</details>

---

### Q3 描述一个死锁场景，以及如何预防？

<details><summary>参考答案</summary>

场景：事务A更新 user_id=1 再更新 user_id=2；事务B反向操作。若并发执行，A等B释放2的锁，B等A释放1的锁 → 死锁。

预防：
1. 统一按主键升序加锁（最有效）
2. 用 `SELECT FOR UPDATE` 在事务开始就锁定所需行
3. 设置 `lock_timeout` 避免无限等待
4. 尽量缩短事务，减少锁持有时间
</details>

---

### Q4 VACUUM 和 VACUUM FULL 的区别？何时用 VACUUM FULL？

<details><summary>参考答案</summary>

- `VACUUM`：将死元组标记为可复用空间，但**不归还给操作系统**。不锁表，可在线执行，适合日常维护。
- `VACUUM FULL`：重写整张表，将死元组空间**归还给 OS**，表文件缩小。需要**排他锁**，期间表不可读写。

VACUUM FULL 适用于：表严重膨胀（如大批量删除后），磁盘空间紧张，且可以安排维护窗口停服。

替代方案：`pg_repack` 可在线重建表（不锁表）。
</details>

---

### Q5 [实战] 如何排查 PostgreSQL 表膨胀问题？

<details><summary>参考答案</summary>

```sql
-- 1. 查看死元组比例
SELECT relname, n_dead_tup, n_live_tup,
       round(n_dead_tup::numeric / NULLIF(n_live_tup,0) * 100, 2) AS dead_pct,
       last_autovacuum, last_autoanalyze
FROM pg_stat_user_tables
WHERE n_dead_tup > 10000
ORDER BY n_dead_tup DESC;

-- 2. 查看表实际大小 vs 预期大小
SELECT relname, pg_size_pretty(pg_total_relation_size(relid)) AS total_size
FROM pg_stat_user_tables
ORDER BY pg_total_relation_size(relid) DESC;

-- 3. 手动触发（如 autovacuum 滞后）
VACUUM ANALYZE 问题表;
```

若 autovacuum 不够及时，考虑调整该表的 `autovacuum_vacuum_scale_factor`。
</details>